In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd drive/MyDrive/Project/dataset

Mounted at /content/drive
/content/drive/MyDrive/Project/dataset


# 抓0050前30檔

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

def fetch_top30_0050():
    url = "https://www.stockfeel.com.tw/%E3%80%90-%E6%9C%80%E6%96%B0%E6%9B%B4%E6%96%B0-%E3%80%910050-%E6%98%AF%E4%BB%80%E9%BA%BC%EF%BC%9F%E6%88%90%E5%88%86%E8%82%A1%E6%9C%89%E5%93%AA%E4%BA%9B%EF%BC%9F/"
    resp = requests.get(url)
    resp.encoding = 'utf-8'
    soup = BeautifulSoup(resp.text, 'html.parser')

    top30 = {}
    # 找出含有成分股的表格：標題通常有「成分股」字樣
    tables = soup.find_all("table")
    index = 1
    for table in tables:
        if "股票代號" in table.get_text() and "股票名稱" in table.get_text():
            rows = table.find_all("tr")[1:31]  # 跳過表頭取前30
            for row in rows:
                cells = row.find_all("td")
                if len(cells) >= 2:
                    code = re.sub(r"\D", "", cells[0].get_text(strip=True))  # 只保留數字部分
                    name = cells[1].get_text(strip=True)
                    if code and name:
                        top30[index] = code + " - " + name
                        index += 1
            break  # 找到就退出

    return top30

if __name__ == "__main__":
    top30_dict = fetch_top30_0050()
    print(top30_dict)


{1: '2330 - 台積電', 2: '2317 - 鴻海', 3: '2454 - 聯發科', 4: '2308 - 台達電', 5: '2382 - 廣達', 6: '2881 - 富邦金', 7: '2891 - 中信金', 8: '2882 - 國泰金', 9: '2303 - 聯電', 10: '2412 - 中華電'}


In [ ]:
import json

with open('top30_0050.json', 'w', encoding='utf-8') as f:
    json.dump(top30_dict, f, ensure_ascii=False, indent=4)

print("top30_0050.json saved successfully.")

top30_0050.json saved successfully.


# Main

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict
import re
from sentence_transformers import SentenceTransformer
import torch
from sklearn.metrics.pairwise import cosine_similarity
import jieba
import warnings
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
warnings.filterwarnings('ignore')

class OptimizedPTTSentimentProcessor:
    def __init__(self, top30_stocks_path='./top30_0050.json', use_gpu=True, batch_size=32):
        self.stock_dict = self.load_top30_stocks(top30_stocks_path)
        self.stock_aliases = self.create_stock_aliases()
        self.batch_size = batch_size

        # GPU設置
        self.device = 'cuda' if use_gpu and torch.cuda.is_available() else 'cpu'
        print(f"使用設備: {self.device}")

        # 載入模型並移至GPU
        print("載入embedding模型...")
        try:
            self.model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=self.device)
            print("✓ 使用多語言模型")
        except:
            try:
                self.model = SentenceTransformer('all-MiniLM-L6-v2', device=self.device)
                print("✓ 使用英文模型")
            except:
                print("⚠️ 無法載入embedding模型")
                self.model = None

        # 預先計算股票embedding
        self.stock_embeddings = self.precompute_stock_embeddings_batch()

        # 預編譯正則表達式
        self.stock_patterns = self.compile_stock_patterns()

        # 情緒關鍵字預處理
        self.positive_keywords, self.negative_keywords = self.prepare_sentiment_keywords()

    def compile_stock_patterns(self):
        """預編譯股票匹配的正則表達式"""
        patterns = {}
        for stock_code, aliases in self.stock_aliases.items():
            # 為每個股票創建一個綜合的正則表達式
            escaped_aliases = [re.escape(alias) for alias in aliases]
            pattern = r'\b(?:' + '|'.join(escaped_aliases + [stock_code]) + r')\b'
            patterns[stock_code] = re.compile(pattern, re.IGNORECASE)
        return patterns

    def prepare_sentiment_keywords(self):
        """預處理情緒關鍵字"""
        positive_keywords = {
            '漲': 2, '上漲': 2, '大漲': 3, '漲停': 3, '飆漲': 3,
            '看好': 1, '利多': 2, '買進': 1, '持有': 1, '加碼': 2,
            '績優': 1, '成長': 1, '突破': 2, '創高': 3, '強勢': 2,
            '獲利': 2, '賺錢': 2, '牛市': 2, '多頭': 2, '看漲': 1,
            '推薦': 1, '優質': 1, '潛力': 1, '機會': 1
        }

        negative_keywords = {
            '跌': 2, '下跌': 2, '大跌': 3, '跌停': 3, '暴跌': 3,
            '看空': 1, '利空': 2, '賣出': 1, '出清': 2, '減碼': 2,
            '套牢': 2, '虧損': 2, '慘跌': 3, '破底': 3, '弱勢': 2,
            '危機': 2, '風險': 1, '熊市': 2, '空頭': 2, '看跌': 1,
            '避開': 1, '小心': 1, '警告': 2, '危險': 2
        }

        return positive_keywords, negative_keywords

    def precompute_stock_embeddings_batch(self):
        """批量計算股票embedding"""
        if not self.model:
            return {}

        all_aliases = []
        alias_to_stock = {}

        for stock_code, aliases in self.stock_aliases.items():
            for alias in aliases:
                all_aliases.append(alias)
                alias_to_stock[alias] = stock_code

        # 批量計算embedding
        print(f"批量計算 {len(all_aliases)} 個股票別名的embedding...")
        try:
            embeddings = self.model.encode(all_aliases, batch_size=self.batch_size, show_progress_bar=True)

            # 整理結果
            stock_embeddings = defaultdict(list)
            for alias, embedding in zip(all_aliases, embeddings):
                stock_code = alias_to_stock[alias]
                stock_embeddings[stock_code].append(embedding)

            # 計算每支股票的平均embedding
            final_embeddings = {}
            for stock_code, emb_list in stock_embeddings.items():
                final_embeddings[stock_code] = np.mean(emb_list, axis=0)

            print(f"✓ 完成 {len(final_embeddings)} 支股票的embedding計算")
            return final_embeddings

        except Exception as e:
            print(f"批量計算embedding失敗: {e}")
            return {}

    def extract_stock_mentions_fast(self, texts):
        """快速批量提取股票提及 - 修正版"""
        batch_results = []

        for text in texts:
            mentioned_stocks = set()

            # 使用預編譯的正則表達式匹配別稱
            for stock_code, pattern in self.stock_patterns.items():
                if pattern.search(text):
                    mentioned_stocks.add(stock_code)

            # 額外檢查純數字股票代碼（4位數字）
            import re
            stock_code_pattern = re.compile(r'\b\d{4}\b')
            found_codes = stock_code_pattern.findall(text)
            for code in found_codes:
                if code in self.stock_dict:
                    mentioned_stocks.add(code)

            batch_results.append(list(mentioned_stocks))

        return batch_results

    def calculate_sentiment_batch(self, texts):
        """批量計算文本情緒"""
        batch_scores = []

        for text in texts:
            sentiment_score = 0

            # 正面情緒
            for keyword, weight in self.positive_keywords.items():
                count = text.count(keyword)
                sentiment_score += count * weight * 0.1

            # 負面情緒
            for keyword, weight in self.negative_keywords.items():
                count = text.count(keyword)
                sentiment_score -= count * weight * 0.1

            # 標準化
            sentiment_score = max(-1, min(1, sentiment_score))
            batch_scores.append(sentiment_score)

        return batch_scores

    def process_posts_batch(self, posts):
        """批量處理文章"""
        if not posts:
            return []

        # 準備批量數據
        texts = []
        titles = []
        contents = []
        all_comments = []

        for post in posts:
            title = post.get('title', '')
            content = post.get('content', '')
            comments = post.get('comments', [])

            titles.append(title)
            contents.append(content)
            texts.append(title + ' ' + content)
            all_comments.append(comments)

        # 批量提取股票提及
        mentioned_stocks_batch = self.extract_stock_mentions_fast(texts)

        # 批量計算文本情緒
        text_sentiments = self.calculate_sentiment_batch(texts)

        # 批量計算embedding情緒（如果需要）
        embedding_sentiments = []
        if self.model and len(texts) > 0:
            try:
                # 限制文本長度以提高效率
                truncated_texts = [text[:200] for text in texts]
                text_embeddings = self.model.encode(truncated_texts, batch_size=self.batch_size)

                # 參考情緒embedding
                pos_emb = self.model.encode(["看好 上漲 獲利"])[0]
                neg_emb = self.model.encode(["看空 下跌 虧損"])[0]

                for text_emb in text_embeddings:
                    pos_sim = cosine_similarity([text_emb], [pos_emb])[0][0]
                    neg_sim = cosine_similarity([text_emb], [neg_emb])[0][0]
                    embedding_sentiment = (pos_sim - neg_sim) * 0.5
                    embedding_sentiments.append(embedding_sentiment)

            except Exception as e:
                print(f"批量embedding計算錯誤: {e}")
                embedding_sentiments = [0] * len(texts)
        else:
            embedding_sentiments = [0] * len(texts)

        # 整合結果
        results = []
        for i, post in enumerate(posts):
            comments = all_comments[i]

            # 推噓文統計
            positive_comments = sum(1 for c in comments if c.get('type') == '推')
            negative_comments = sum(1 for c in comments if c.get('type') == '噓')
            total_comments = len(comments)

            comment_sentiment = (positive_comments - negative_comments) / total_comments if total_comments > 0 else 0

            # 綜合情緒
            final_sentiment = (comment_sentiment * 0.5 +
                             text_sentiments[i] * 0.3 +
                             embedding_sentiments[i] * 0.2)

            result = {
                'mentioned_stocks': mentioned_stocks_batch[i],
                'sentiment_data': {
                    'comment_sentiment': comment_sentiment,
                    'text_sentiment': text_sentiments[i],
                    'embedding_sentiment': embedding_sentiments[i],
                    'final_sentiment': final_sentiment,
                    'engagement_score': total_comments,
                    'push_count': positive_comments,
                    'boo_count': negative_comments,
                    'neutral_count': total_comments - positive_comments - negative_comments
                },
                'content_length': len(titles[i]) + len(contents[i])
            }
            results.append(result)

        return results

    def process_daily_data_optimized(self, json_data, target_date):
        """優化版處理單日資料"""
        if not json_data:
            return self.create_empty_daily_record(target_date)

        # 批量處理所有文章
        batch_results = self.process_posts_batch(json_data)

        # 統計結果
        daily_stats = {
            'date': target_date,
            'post_count': len(json_data),
            'total_engagement': 0,
            'avg_sentiment': 0,
            'positive_posts': 0,
            'negative_posts': 0,
            'neutral_posts': 0,
            'market_sentiment': 0,
            'content_volume': 0,
            'sentiment_volatility': 0
        }

        # 初始化股票統計
        for stock_code in self.stock_dict.keys():
            daily_stats[f'mentions_{stock_code}'] = 0
            daily_stats[f'sentiment_{stock_code}'] = 0

        sentiment_scores = []
        stock_sentiments = defaultdict(list)
        weighted_sentiments = []

        for result in batch_results:
            sentiment_data = result['sentiment_data']
            mentioned_stocks = result['mentioned_stocks']

            sentiment_scores.append(sentiment_data['final_sentiment'])
            daily_stats['total_engagement'] += sentiment_data['engagement_score']
            daily_stats['content_volume'] += result['content_length']

            # 分類情緒
            if sentiment_data['final_sentiment'] > 0.1:
                daily_stats['positive_posts'] += 1
            elif sentiment_data['final_sentiment'] < -0.1:
                daily_stats['negative_posts'] += 1
            else:
                daily_stats['neutral_posts'] += 1

            # 股票統計
            for stock in mentioned_stocks:
                daily_stats[f'mentions_{stock}'] += 1
                stock_sentiments[stock].append(sentiment_data['final_sentiment'])

            # 加權情緒
            weight = max(1, sentiment_data['engagement_score'])
            weighted_sentiments.extend([sentiment_data['final_sentiment']] * weight)

        # 計算最終統計
        if sentiment_scores:
            daily_stats['avg_sentiment'] = np.mean(sentiment_scores)
            daily_stats['sentiment_volatility'] = np.std(sentiment_scores)

        if weighted_sentiments:
            daily_stats['market_sentiment'] = np.mean(weighted_sentiments)

        # 各股票平均情緒
        for stock_code in self.stock_dict.keys():
            if stock_sentiments[stock_code]:
                daily_stats[f'sentiment_{stock_code}'] = np.mean(stock_sentiments[stock_code])

        return daily_stats

    def process_file_worker(self, args):
        """多進程工作函數"""
        date_str, data_folder = args
        json_file = os.path.join(data_folder, f'ptt_{date_str}.json')

        if os.path.exists(json_file):
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    daily_json = json.load(f)

                daily_stats = self.process_daily_data_optimized(daily_json, date_str)
                return date_str, daily_stats, len(daily_json)

            except Exception as e:
                print(f"✗ {date_str}: 處理錯誤 - {e}")
                return date_str, self.create_empty_daily_record(date_str), 0
        else:
            return date_str, self.create_empty_daily_record(date_str), 0

    def process_all_daily_data_parallel(self, start_date, end_date, data_folder='./archive', max_workers=None):
        """並行處理所有日期資料"""
        start = datetime.strptime(start_date, '%Y-%m-%d')
        end = datetime.strptime(end_date, '%Y-%m-%d')

        # 準備日期列表
        date_args = []
        current_date = start
        while current_date <= end:
            date_str = current_date.strftime('%Y-%m-%d')
            date_args.append((date_str, data_folder))
            current_date += timedelta(days=1)

        print(f"開始並行處理 {len(date_args)} 天的資料...")

        # 使用線程池處理（因為主要是I/O操作）
        if max_workers is None:
            max_workers = min(8, mp.cpu_count())

        results = {}
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(self.process_file_worker, args): args[0] for args in date_args}

            for future in futures:
                date_str = futures[future]
                try:
                    date_str, daily_stats, post_count = future.result()
                    results[date_str] = daily_stats

                    if post_count > 0:
                        sentiment = daily_stats.get('market_sentiment', 0)
                        print(f"✓ {date_str}: {post_count} 篇文章, 情緒: {sentiment:.3f}")
                    else:
                        print(f"- {date_str}: 無資料")

                except Exception as e:
                    print(f"✗ {date_str}: 處理失敗 - {e}")
                    results[date_str] = self.create_empty_daily_record(date_str)

        # 按日期排序
        sorted_results = []
        current_date = start
        while current_date <= end:
            date_str = current_date.strftime('%Y-%m-%d')
            sorted_results.append(results[date_str])
            current_date += timedelta(days=1)

        return sorted_results

    def load_top30_stocks(self, file_path):
        """載入前30大股票資料"""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                top30_data = json.load(f)

            stocks_dict = {}
            for rank, stock_info in top30_data.items():
                parts = stock_info.split(' - ')
                if len(parts) == 2:
                    stock_code = parts[0].strip()
                    company_name = parts[1].strip()
                    stocks_dict[stock_code] = company_name

            print(f"載入 {len(stocks_dict)} 支股票")
            return stocks_dict
        except Exception as e:
            print(f"載入股票資料錯誤: {e}")
            return {}

    def create_stock_aliases(self):
        """創建股票別稱字典 - 修正版"""
        # 預定義的特殊別稱
        special_aliases = {
            '2330': ['台積電', 'TSMC', '積電', '護國神山', '神山', 'TSM', '台積', '晶圓代工龍頭'],
            '2317': ['鴻海', '富士康', 'Foxconn', '海公公', '鴻海精密', '代工王', 'HON HAI'],
            '2454': ['聯發科', 'MTK', 'MediaTek', '發哥', '聯發', 'IC設計龍頭'],
            '2308': ['台達電', 'Delta', '台達', '電源供應器龍頭'],
            '2382': ['廣達', 'Quanta', '廣達電腦', '筆電代工'],
            '2881': ['富邦金', '富邦', 'Fubon', '富邦金控'],
            '2891': ['中信金', '中國信託', 'CTBC', '中信', '中信金控'],
            '2882': ['國泰金', '國泰', 'Cathay', '國泰金控'],
            '2303': ['聯電', 'UMC', '聯華電子', 'United Microelectronics'],
            '2412': ['中華電', '中華電信', 'Chunghwa', 'CHT', '電信龍頭']
        }

        aliases = {}

        # 為所有股票創建別稱
        for code, name in self.stock_dict.items():
            stock_aliases = [code, name]
            if code in special_aliases:
                stock_aliases.extend(special_aliases[code])
            for suffix in ['股份有限公司', '有限公司', '公司', '集團', '控股', '投資']:
                if name.endswith(suffix):
                    short_name = name.replace(suffix, '')
                    if short_name and short_name not in stock_aliases:
                        stock_aliases.append(short_name)
            if '電' in name:
                short_name = name.replace('電', '')
                if short_name and short_name not in stock_aliases:
                    stock_aliases.append(short_name)
            if '金' in name and '金控' in name:
                short_name = name.replace('金控', '').replace('金', '')
                if short_name and short_name not in stock_aliases:
                    stock_aliases.append(short_name)
            if len(name) >= 2:
                key_name = name[:2]
                if key_name not in stock_aliases:
                    stock_aliases.append(key_name)
            if len(name) >= 3:
                key_name = name[:3]
                if key_name not in stock_aliases:
                    stock_aliases.append(key_name)
            aliases[code] = list(set([alias for alias in stock_aliases if alias.strip()]))
        return aliases

    def create_empty_daily_record(self, date_str):
        """創建空日期記錄"""
        record = {
            'date': date_str,
            'post_count': 0,
            'total_engagement': 0,
            'avg_sentiment': np.nan,
            'positive_posts': 0,
            'negative_posts': 0,
            'neutral_posts': 0,
            'market_sentiment': np.nan,
            'content_volume': 0,
            'sentiment_volatility': np.nan
        }

        for stock_code in self.stock_dict.keys():
            record[f'mentions_{stock_code}'] = 0
            record[f'sentiment_{stock_code}'] = np.nan

        return record

    def fill_missing_data(self, daily_features, method='exp_mean', window=7):
        """填補缺失資料"""
        df = pd.DataFrame(daily_features)
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)

        sentiment_columns = [col for col in df.columns
                           if 'sentiment' in col or col in ['avg_sentiment', 'sentiment_volatility']]

        for col in sentiment_columns:
            if method == 'moving_average':
                df[col] = df[col].fillna(df[col].rolling(window=window, min_periods=1).mean())
            elif method == 'exp_mean':
                df[col] = df[col].fillna(df[col].ewm(span=window, adjust=False).mean())

            df[col] = df[col].fillna(0)

        return df

    def export_results(self, df, output_path='./ptt_sentiment_features_optimized.csv'):
        """匯出結果"""
        df['sentiment_ma3'] = df['market_sentiment'].rolling(3).mean()
        df['sentiment_ma7'] = df['market_sentiment'].rolling(7).mean()
        df['sentiment_change'] = df['market_sentiment'].pct_change()
        df['market_heat'] = df['total_engagement']
        df['heat_ma7'] = df['market_heat'].rolling(7).mean()

        df = df.fillna(method='bfill').fillna(method='ffill').fillna(0)

        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\n資料已匯出至: {output_path}")
        print(f"資料維度: {df.shape}")

        return df

# 使用優化版本
if __name__ == "__main__":
    # 創建優化處理器
    processor = OptimizedPTTSentimentProcessor(
        './top30_0050.json',
        use_gpu=True,  # 使用GPU
        batch_size=64  # 增大批次大小
    )

    # 並行處理
    daily_features = processor.process_all_daily_data_parallel(
        '2020-01-01', '2025-08-07',
        max_workers=8  # 調整並行數量
    )

    df = processor.fill_missing_data(daily_features, method='exp_mean', window=7)
    final_df = processor.export_results(df)

    print(f"\n處理完成！")
    print(f"總計處理: {len(final_df)} 天")
    print(f"平均每日文章數: {final_df['post_count'].mean():.1f}")
    print(f"平均市場情緒: {final_df['market_sentiment'].mean():.3f}")

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-62668350.py", line 8, in <cell line: 0>
    from sentence_transformers import SentenceTransformer
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1138, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1078, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1507, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1476, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1434, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshel

# Read final_df

In [ ]:
import pandas as pd
df = pd.read_csv("./ptt_sentiment_features.csv")

In [ ]:
df.head()

,date,post_count,total_engagement,avg_sentiment,positive_posts,negative_posts,neutral_posts,market_sentiment,content_volume,sentiment_volatility,...,sentiment_2882,mentions_2303,sentiment_2303,mentions_2412,sentiment_2412,sentiment_ma3,sentiment_ma7,sentiment_change,market_heat,heat_ma7
0,2020-01-01,19,1084,0.370415,17,2,0,0.386182,31041,0.221054,...,0.000000,0,0.000000,0,0.000000,0.318134,0.281946,-0.402465,1084,3234.857143
1,2020-01-02,43,3581,0.350561,40,0,3,0.230757,44536,0.173028,...,0.557282,1,0.347838,1,0.347838,0.318134,0.281946,-0.402465,3581,3234.857143
2,2020-01-03,42,4868,0.325525,38,0,4,0.337462,36158,0.173227,...,0.000000,0,0.000000,0,0.000000,0.318134,0.281946,0.462413,4868,3234.857143
3,2020-01-04,17,1155,0.259754,13,0,4,0.250957,18414,0.167092,...,0.319839,1,0.319839,0,0.000000,0.273059,0.281946,-0.256340,1155,3234.857143
4,2020-01-05,27,1395,0.213067,22,3,2,0.229419,26301,0.201832,...,0.000000,0,0.000000,0,0.000000,0.272613,0.281946,-0.085823,1395,3234.857143


In [ ]:
df.columns

Index(['date', 'post_count', 'total_engagement', 'avg_sentiment',
       'positive_posts', 'negative_posts', 'neutral_posts', 'market_sentiment',
       'content_volume', 'sentiment_volatility', 'mentions_2330',
       'sentiment_2330', 'mentions_2317', 'sentiment_2317', 'mentions_2454',
       'sentiment_2454', 'mentions_2308', 'sentiment_2308', 'mentions_2382',
       'sentiment_2382', 'mentions_2881', 'sentiment_2881', 'mentions_2891',
       'sentiment_2891', 'mentions_2882', 'sentiment_2882', 'mentions_2303',
       'sentiment_2303', 'mentions_2412', 'sentiment_2412', 'mentions_2884',
       'sentiment_2884', 'mentions_2886', 'sentiment_2886', 'mentions_3711',
       'sentiment_3711', 'mentions_2357', 'sentiment_2357', 'mentions_1216',
       'sentiment_1216', 'mentions_2885', 'sentiment_2885', 'mentions_2345',
       'sentiment_2345', 'mentions_3231', 'sentiment_3231', 'mentions_3034',
       'sentiment_3034', 'mentions_2892', 'sentiment_2892', 'mentions_2379',
       'sentiment_

In [ ]:
import os
import pandas as pd
import re
os.makedirs("./PTT", exist_ok=True)

# 基本欄位（所有股票檔案都會包含）
base_columns = ['date', 'post_count', 'total_engagement', 'avg_sentiment',
                'positive_posts', 'negative_posts', 'neutral_posts', 'market_sentiment',
                'content_volume', 'sentiment_volatility', 'sentiment_ma3',
                'sentiment_ma7', 'sentiment_change', 'market_heat', 'heat_ma7']

# 找出所有股票代碼
stock_codes = set()
for col in df.columns:
    if col.startswith('mentions_') or col.startswith('sentiment_'):
        # 提取股票代碼
        match = re.search(r'_(\d+)$', col)
        if match:
            stock_codes.add(match.group(1))

print(f"找到的股票代碼: {sorted(stock_codes)}")

# 為每個股票創建獨立的CSV檔案
for stock_code in stock_codes:
    # 該股票的特定欄位
    stock_columns = [f'mentions_{stock_code}', f'sentiment_{stock_code}']

    # 檢查這些欄位是否存在於原始資料中
    existing_stock_columns = [col for col in stock_columns if col in df.columns]

    # 組合最終的欄位列表
    final_columns = base_columns + existing_stock_columns

    # 確保所有欄位都存在於原始資料中
    available_columns = [col for col in final_columns if col in df.columns]

    # 創建該股票的資料框
    stock_df = df[available_columns].copy()

    # 儲存為CSV檔案
    filename = f'./PTT/PTT_{stock_code}.csv'
    stock_df.to_csv(filename, index=False, encoding='utf-8-sig')

    print(f"已儲存 {filename}，包含欄位: {len(available_columns)} 個")
    print(f"股票特定欄位: {existing_stock_columns}")
    print("-" * 50)

找到的股票代碼: ['1216', '2303', '2308', '2317', '2330', '2345', '2357', '2379', '2382', '2383', '2412', '2454', '2880', '2881', '2882', '2883', '2884', '2885', '2886', '2890', '2891', '2892', '3008', '3017', '3034', '3231', '3661', '3711', '5880', '6669']
已儲存 ./PTT/PTT_3231.csv，包含欄位: 17 個
股票特定欄位: ['mentions_3231', 'sentiment_3231']
--------------------------------------------------
已儲存 ./PTT/PTT_1216.csv，包含欄位: 17 個
股票特定欄位: ['mentions_1216', 'sentiment_1216']
--------------------------------------------------
已儲存 ./PTT/PTT_2885.csv，包含欄位: 17 個
股票特定欄位: ['mentions_2885', 'sentiment_2885']
--------------------------------------------------
已儲存 ./PTT/PTT_2454.csv，包含欄位: 17 個
股票特定欄位: ['mentions_2454', 'sentiment_2454']
--------------------------------------------------
已儲存 ./PTT/PTT_3008.csv，包含欄位: 17 個
股票特定欄位: ['mentions_3008', 'sentiment_3008']
--------------------------------------------------
已儲存 ./PTT/PTT_2345.csv，包含欄位: 17 個
股票特定欄位: ['mentions_2345', 'sentiment_2345']
---------------------------

# 基本統計欄位

## 📅 時間與數量

- date: 資料日期 (YYYY-MM-DD格式)

- post_count: 當日PTT文章總數

- total_engagement: 當日總互動數 (所有推噓文數量加總)

- content_volume: 當日內容總字數 (標題+內文字數)

## 😊😐😞 情緒分類統計

- positive_posts: 正面情緒文章數 (情緒分數 > 0.1)

- negative_posts: 負面情緒文章數 (情緒分數 < -0.1)

- neutral_posts: 中性情緒文章數 (情緒分數介於 -0.1 到 0.1)

## 📊 情緒指標

- avg_sentiment: 當日平均情緒分數 (-1到1之間，越高越正面)

- market_sentiment: 市場整體情緒 (以互動數加權的情緒分數)

- sentiment_volatility: 情緒波動度 (情緒分數的標準差)

# 個股相關欄位

每支股票都有兩個對應欄位：

## 🏢 個股提及與情緒
- mentions_XXXX: 該股票當日被提及次數
- sentiment_XXXX: 該股票當日平均情緒分數

# 技術指標欄位

## 📈 移動平均與趨勢

- sentiment_ma3: 3日情緒移動平均 (短期趨勢)

- sentiment_ma7: 7日情緒移動平均 (中期趨勢)

- sentiment_change: 情緒變化率 (當日vs前日的百分比變化)

## 🔥 市場熱度指標

- market_heat: 市場熱度 (等同於total_engagement)

- heat_ma7: 7日熱度移動平均 (市場關注度趨勢)